# Phase 3 — NLP & Transformers
## Day 13: Hugging Face Basics
**Date:** 2026-04-24

### Learning Objectives
- Use `pipeline()` for quick NLP tasks (sentiment, NER, summarization, etc.)
- Understand `AutoTokenizer` and how tokenization works
- Load pretrained models with `AutoModel` and `AutoModelForSequenceClassification`
- See how tokenizer output maps to model input
- Know when to use `pipeline()` vs manual tokenizer+model

In [ ]:
# Setup: install transformers if needed
# !pip install transformers torch --quiet

import numpy as np
import torch
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
)

print(f"torch version: {torch.__version__}")
import transformers
print(f"transformers version: {transformers.__version__}")

---
## 1. The `pipeline()` Function: NLP in One Line

Hugging Face's `pipeline()` is the fastest way to use a pretrained model. It handles tokenization, model inference, and post-processing all in one call. You just pick a task and pass in text.

Supported tasks include: `sentiment-analysis`, `ner`, `question-answering`, `summarization`, `translation`, `text-generation`, `fill-mask`, `zero-shot-classification`, and more.

In [ ]:
# Sentiment analysis with one line
classifier = pipeline("sentiment-analysis")

result = classifier("I love learning about transformers!")
print("Sentiment:", result)

# Try multiple texts at once
results = classifier([
    "This movie was absolutely fantastic!",
    "The weather is terrible today.",
    "I had a regular day at work."
])

for text, res in zip(["fantastic movie", "terrible weather", "regular day"], results):
    print(f"  {text:20s} -> {res['label']:8s} ({res['score']:.4f})")

In [ ]:
# Fill-mask: BERT's original training task
# The model predicts what word belongs in the [MASK] position

unmasker = pipeline("fill-mask")

results = unmasker("The capital of France is [MASK].")
print("Fill in the blank: 'The capital of France is [MASK].'")
for r in results[:5]:
    print(f"  {r['token_str']:12s} (score: {r['score']:.4f})")

In [ ]:
# Named Entity Recognition (NER)
# Finds people, organizations, locations, etc. in text

ner = pipeline("ner", grouped_entities=True)

text = "Elon Musk founded SpaceX in Hawthorne, California."
entities = ner(text)

print(f"Text: {text}")
print("\nEntities found:")
for ent in entities:
    print(f"  {ent['word']:20s} -> {ent['entity_group']:5s} (score: {ent['score']:.4f})")

In [ ]:
# Zero-shot classification: classify text into categories the model was never trained on
# This is surprisingly powerful

zero_shot = pipeline("zero-shot-classification")

result = zero_shot(
    "The stock market dropped 5% after the Federal Reserve raised interest rates.",
    candidate_labels=["politics", "finance", "sports", "technology"]
)

print("Text: stock market dropped after Fed raised rates")
print("\nPredicted labels:")
for label, score in zip(result['labels'], result['scores']):
    bar = '|' * int(score * 30)
    print(f"  {label:12s} {score:.4f} {bar}")

In [ ]:
# Question answering: extract answers from a context paragraph

qa = pipeline("question-answering")

context = """Hugging Face was founded in 2016 by Clement Delangue, Julien Chaumond, 
and Thomas Wolf. The company is headquartered in New York City. They created the 
Transformers library which has become the most popular NLP library in the world."""

questions = [
    "When was Hugging Face founded?",
    "Where is Hugging Face headquartered?",
    "What library did they create?"
]

for q in questions:
    answer = qa(question=q, context=context)
    print(f"Q: {q}")
    print(f"A: {answer['answer']} (score: {answer['score']:.4f})")
    print()

---
## 2. AutoTokenizer: Turning Text into Numbers

Before a model can process text, it needs to be converted into numbers. This is what a tokenizer does. Different models use different tokenization strategies:

- **WordPiece** (BERT): splits unknown words into subwords. "unhappiness" becomes ["un", "##happi", "##ness"]
- **BPE** (GPT-2, RoBERTa): similar idea, different algorithm
- **SentencePiece** (T5, XLNet): can handle any language without pre-tokenization

`AutoTokenizer.from_pretrained()` loads the right tokenizer for any model. You don't need to know which strategy it uses.

In [ ]:
# Load BERT's tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Hugging Face makes NLP easy!"

# Tokenize
tokens = tokenizer.tokenize(text)
print(f"Original text: {text}")
print(f"Tokens:        {tokens}")
print(f"Number of tokens: {len(tokens)}")

In [ ]:
# tokenize() just splits. encode() also converts to IDs and adds special tokens.
# __call__ (the recommended way) does everything at once.

# Method 1: tokenize + convert
tokens = tokenizer.tokenize(text)
ids = tokenizer.convert_tokens_to_ids(tokens)
print("Step by step:")
print(f"  Tokens: {tokens}")
print(f"  IDs:    {ids}")

# Method 2: encode (adds special tokens)
ids_with_special = tokenizer.encode(text)
print(f"\nencode() output: {ids_with_special}")
print(f"  Decoded: {tokenizer.decode(ids_with_special)}")
print(f"  Note: [CLS] at start (ID {ids_with_special[0]}) and [SEP] at end (ID {ids_with_special[-1]})")

# Method 3: __call__ (recommended, returns a dict with everything the model needs)
encoded = tokenizer(text, return_tensors="pt")  # pt = PyTorch tensors
print(f"\n__call__ output keys: {list(encoded.keys())}")
print(f"  input_ids:      {encoded['input_ids']}")
print(f"  attention_mask:  {encoded['attention_mask']}")

In [ ]:
# WordPiece tokenization: how BERT handles unknown words
# It breaks them into known subwords

examples = [
    "unhappiness",
    "transformers",
    "antidisestablishmentarianism",
    "ChatGPT",
    "tokenization",
]

print("How BERT tokenizes different words:")
for word in examples:
    tokens = tokenizer.tokenize(word)
    print(f"  {word:35s} -> {tokens}")

print("\nTokens starting with ## are continuations of the previous token.")
print("This lets BERT handle any word, even ones it has never seen before.")

In [ ]:
# Special tokens: [CLS], [SEP], [PAD], [MASK], [UNK]
print("BERT's special tokens:")
print(f"  [CLS]  = {tokenizer.cls_token_id:6d}  Start of every input")
print(f"  [SEP]  = {tokenizer.sep_token_id:6d}  Separates sentences / end of input")
print(f"  [PAD]  = {tokenizer.pad_token_id:6d}  Padding for batch processing")
print(f"  [MASK] = {tokenizer.mask_token_id:6d}  Used in masked language modeling")
print(f"  [UNK]  = {tokenizer.unk_token_id:6d}  Unknown token (rare with WordPiece)")
print(f"\n  Vocab size: {tokenizer.vocab_size:,}")

In [ ]:
# Padding and truncation: making batches of different-length texts
texts = [
    "Short text.",
    "This is a medium length sentence about NLP.",
    "And this is a longer sentence that contains more words and should generate more tokens."
]

# Without padding: each input has different length
for t in texts:
    enc = tokenizer(t)
    print(f"  '{t[:30]:30s}...' -> {len(enc['input_ids'])} tokens")

# With padding: all inputs get the same length
batch = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
print(f"\nBatched shape: {batch['input_ids'].shape}")
print(f"All padded to length {batch['input_ids'].shape[1]}")
print(f"\nAttention mask (1=real token, 0=padding):")
for i, t in enumerate(texts):
    mask = batch['attention_mask'][i].tolist()
    print(f"  {mask}")

---
## 3. AutoModel: Loading Pretrained Models

Just like `AutoTokenizer` picks the right tokenizer, `AutoModel` picks the right model architecture. You give it a model name from the Hugging Face Hub, and it downloads the weights.

There are specialized classes for different tasks:
- `AutoModel`: base model, outputs raw hidden states
- `AutoModelForSequenceClassification`: adds a classification head on top
- `AutoModelForTokenClassification`: for NER-style tasks
- `AutoModelForQuestionAnswering`: for QA tasks
- `AutoModelForMaskedLM`: for fill-mask tasks

In [ ]:
# Load the base BERT model
model = AutoModel.from_pretrained("bert-base-uncased")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"BERT-base parameters: {total_params:,}")
print(f"That's about {total_params / 1e6:.0f} million parameters")

# Model architecture overview
print(f"\nModel type: {model.config.model_type}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Attention heads: {model.config.num_attention_heads}")
print(f"Layers: {model.config.num_hidden_layers}")
print(f"Vocab size: {model.config.vocab_size}")

In [ ]:
# Forward pass: text -> tokens -> model -> hidden states

text = "Transformers changed NLP forever."
inputs = tokenizer(text, return_tensors="pt")

print("Input to model:")
print(f"  input_ids shape:      {inputs['input_ids'].shape}")
print(f"  attention_mask shape:  {inputs['attention_mask'].shape}")

# Run through model (no gradient needed for inference)
with torch.no_grad():
    outputs = model(**inputs)

print(f"\nOutput keys: {list(outputs.keys())}")
print(f"last_hidden_state shape: {outputs.last_hidden_state.shape}")
print("  -> (batch_size=1, seq_len, hidden_size=768)")
print(f"\nEach token now has a 768-dimensional context-aware representation.")

# The [CLS] token's representation is often used for classification
cls_embedding = outputs.last_hidden_state[0, 0, :]
print(f"\n[CLS] embedding (first 10 values): {cls_embedding[:10].tolist()}")

In [ ]:
# AutoModelForSequenceClassification: model with a classification head
# This is what you use for sentiment analysis, topic classification, etc.

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
clf_tokenizer = AutoTokenizer.from_pretrained(model_name)
clf_model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Check the label mapping
print("Label mapping:", clf_model.config.id2label)

# Run inference manually (this is what pipeline() does internally)
texts = [
    "I absolutely loved this movie!",
    "This was the worst experience of my life.",
]

inputs = clf_tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

with torch.no_grad():
    outputs = clf_model(**inputs)

# Raw logits -> probabilities via softmax
logits = outputs.logits
probs = torch.nn.functional.softmax(logits, dim=-1)

print("\nManual classification results:")
for i, text in enumerate(texts):
    pred_label = clf_model.config.id2label[probs[i].argmax().item()]
    confidence = probs[i].max().item()
    print(f"  '{text[:40]:40s}' -> {pred_label} ({confidence:.4f})")

---
## 4. Understanding Model Outputs

Different model classes return different things:

- **AutoModel**: returns `last_hidden_state` (one vector per token) and optionally `pooler_output` (processed [CLS] token)
- **AutoModelForSequenceClassification**: returns `logits` (one score per class)
- **AutoModelForTokenClassification**: returns `logits` per token

The raw outputs are logits (unnormalized scores). Apply softmax to get probabilities.

In [ ]:
# Comparing base model vs classification model output

text = "Machine learning is fascinating."

# Base model output
base_inputs = tokenizer(text, return_tensors="pt")
with torch.no_grad():
    base_out = model(**base_inputs)

print("BASE MODEL output:")
print(f"  last_hidden_state: {base_out.last_hidden_state.shape}")
print(f"  -> One 768-dim vector per token")
if hasattr(base_out, 'pooler_output') and base_out.pooler_output is not None:
    print(f"  pooler_output:     {base_out.pooler_output.shape}")
    print(f"  -> Single 768-dim vector for the whole sequence")

# Classification model output
clf_inputs = clf_tokenizer(text, return_tensors="pt")
with torch.no_grad():
    clf_out = clf_model(**clf_inputs)

print(f"\nCLASSIFICATION MODEL output:")
print(f"  logits: {clf_out.logits.shape}")
print(f"  -> One score per class: {clf_out.logits}")
print(f"  -> After softmax: {torch.nn.functional.softmax(clf_out.logits, dim=-1)}")

In [ ]:
# Token-level embeddings: each token gets its own representation
# These are context-dependent (unlike Word2Vec)

sentence1 = "I went to the bank to deposit money."
sentence2 = "I sat on the bank of the river."

enc1 = tokenizer(sentence1, return_tensors="pt")
enc2 = tokenizer(sentence2, return_tensors="pt")

with torch.no_grad():
    out1 = model(**enc1)
    out2 = model(**enc2)

# Find the position of "bank" in each sentence
tokens1 = tokenizer.tokenize(sentence1)
tokens2 = tokenizer.tokenize(sentence2)

bank_idx1 = tokens1.index("bank") + 1  # +1 for [CLS]
bank_idx2 = tokens2.index("bank") + 1

bank_emb1 = out1.last_hidden_state[0, bank_idx1]
bank_emb2 = out2.last_hidden_state[0, bank_idx2]

# Cosine similarity between the two "bank" embeddings
cos_sim = torch.nn.functional.cosine_similarity(bank_emb1.unsqueeze(0), bank_emb2.unsqueeze(0))

print(f"Sentence 1: {sentence1}")
print(f"Sentence 2: {sentence2}")
print(f"\nCosine similarity of 'bank' in both contexts: {cos_sim.item():.4f}")
print("\nBERT gives 'bank' DIFFERENT embeddings depending on context.")
print("Word2Vec would give the same vector regardless of context.")

---
## 5. Choosing Models from the Hugging Face Hub

The Hugging Face Hub (huggingface.co/models) has thousands of pretrained models. Here's how to pick the right one:

**By size**: `bert-base` (110M params) vs `bert-large` (340M params). Larger = better but slower.

**By variant**: `distilbert` (66M params, 97% of BERT's accuracy, 60% faster). Great for production.

**By language**: `bert-base-multilingual-cased` for multilingual, `dbmdz/bert-base-turkish-cased` for Turkish.

**By task**: Look for models already fine-tuned on your task.

In [ ]:
# Compare tokenizers from different models
model_names = [
    "bert-base-uncased",
    "bert-base-cased",
    "distilbert-base-uncased",
]

text = "The Quick Brown Fox Jumps Over The Lazy Dog"

print(f"Text: {text}\n")
for name in model_names:
    tok = AutoTokenizer.from_pretrained(name)
    tokens = tok.tokenize(text)
    print(f"{name:35s} -> {tokens}")

print("\nNotice: uncased models lowercase everything.")
print("Cased models preserve capitalization.")

In [ ]:
# pipeline() with a specific model

fast_classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

test_texts = [
    "The food was absolutely delicious!",
    "I waited 2 hours and the service was rude.",
    "The restaurant was okay, nothing special.",
]

print("Sentiment analysis with DistilBERT:")
results = fast_classifier(test_texts)
for text, res in zip(test_texts, results):
    print(f"  {res['label']:8s} ({res['score']:.4f}) | {text}")

---
## 6. pipeline() vs Manual Approach

When should you use `pipeline()` and when should you use the manual `tokenizer + model` approach?

**Use `pipeline()` when:** quick prototyping, standard tasks, you don't need to customize.

**Use manual approach when:** you need hidden states, you're fine-tuning (Day 14), you want embeddings, or you need custom pre/post-processing.

In [ ]:
# Side-by-side: pipeline vs manual for sentiment analysis

text = "Hugging Face makes NLP so much easier!"

# Approach 1: pipeline (easy)
pipe = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
pipe_result = pipe(text)
print("Pipeline result:", pipe_result)

# Approach 2: manual (full control)
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")
mdl = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

inputs = tok(text, return_tensors="pt", truncation=True)
with torch.no_grad():
    logits = mdl(**inputs).logits

probs = torch.nn.functional.softmax(logits, dim=-1)
pred_id = probs.argmax().item()
pred_label = mdl.config.id2label[pred_id]
confidence = probs[0, pred_id].item()

print(f"Manual result:   label='{pred_label}', score={confidence:.4f}")
print("\nSame result! Pipeline just wraps these steps for convenience.")

---
## Tricky Bits

Common gotchas when working with Hugging Face models.

In [ ]:
# Tricky Bit 1: Tokenizer and model must match!

text = "Hello"

tok_a = AutoTokenizer.from_pretrained("bert-base-uncased")
tok_b = AutoTokenizer.from_pretrained("bert-base-cased")

ids_a = tok_a.encode(text)
ids_b = tok_b.encode(text)
print(f"Same word '{text}', different tokenizers:")
print(f"  bert-base-uncased IDs: {ids_a}")
print(f"  bert-base-cased IDs:   {ids_b}")
print(f"  Different IDs! Using the wrong tokenizer = wrong input to model.")

In [ ]:
# Tricky Bit 2: Token count != Word count
# Models have a max length (usually 512 for BERT)

short_text = "Hello world"
long_word = "supercalifragilisticexpialidocious"

tok = AutoTokenizer.from_pretrained("bert-base-uncased")

print(f"'{short_text}': {len(short_text.split())} words, {len(tok.tokenize(short_text))} tokens")
print(f"'{long_word}': 1 word, {len(tok.tokenize(long_word))} tokens")
print(f"\nMax sequence length for BERT: {tok.model_max_length}")

# What happens with very long text?
long_text = "word " * 1000  # 1000 words
encoded = tok(long_text, truncation=True, max_length=512)
print(f"\n1000-word text truncated to {len(encoded['input_ids'])} tokens")
print("Always use truncation=True to avoid crashes!")

In [ ]:
# Tricky Bit 3: return_tensors matters

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
text = "Test sentence"

# Without return_tensors: Python lists
encoded_list = tok(text)
print(f"Without return_tensors: {type(encoded_list['input_ids'])}")

# With return_tensors="pt": PyTorch tensors
encoded_pt = tok(text, return_tensors="pt")
print(f"With return_tensors='pt': {type(encoded_pt['input_ids'])}")
print(f"  Shape: {encoded_pt['input_ids'].shape}")

# The model needs tensors, not lists!
print("\nAlways use return_tensors='pt' when passing to a PyTorch model.")

---
## Trick Questions

**Q1:** If BERT's tokenizer splits "playing" into ["play", "##ing"], does the model see two separate words?

<details>
<summary>Answer</summary>
No. The model sees two tokens, not two words. The ## prefix tells us "ing" is a continuation of "play". The model processes both through self-attention, so it effectively learns to combine them.
</details>

**Q2:** Can you use a BERT tokenizer with a GPT-2 model?

<details>
<summary>Answer</summary>
Technically yes, but results would be garbage. Each tokenizer has its own vocabulary. Token ID 5000 in BERT maps to a different word than in GPT-2. Always use the matching tokenizer.
</details>

**Q3:** What does the attention_mask do in a padded batch?

<details>
<summary>Answer</summary>
It tells the model which tokens are real (1) and which are padding (0). Without it, the model would attend to padding tokens, corrupting the representations.
</details>

**Q4:** Why does `pipeline()` sometimes give different results than the manual approach?

<details>
<summary>Answer</summary>
Usually because of different preprocessing. pipeline() might truncate or pad differently. Also, if you don't specify a model, pipeline() picks a default. Always specify the model explicitly for reproducibility.
</details>

**Q5:** What is DistilBERT and why would you use it?

<details>
<summary>Answer</summary>
DistilBERT is a smaller, faster version of BERT (66M vs 110M params) created through knowledge distillation. It retains 97% of BERT's performance while running 60% faster. Great for production or limited compute.
</details>

---
## Exercises

Fill in the `___` blanks. The `assert` statements check your work.

In [ ]:
# Exercise 1: Create a sentiment analysis pipeline

my_pipeline = ___("sentiment-analysis")

result = my_pipeline("I love pizza!")
assert result[0]['label'] in ['POSITIVE', 'NEGATIVE']
assert 0 <= result[0]['score'] <= 1
print(f"Exercise 1 passed! Result: {result}")

In [ ]:
# Exercise 2: Load a tokenizer and tokenize a sentence

tok_ex2 = AutoTokenizer.from_pretrained("bert-base-uncased")
tokens = tok_ex2.___("Hello world")  # which method splits text into tokens?

assert isinstance(tokens, list)
assert len(tokens) == 2  # "hello" and "world" (lowercased by uncased model)
print(f"Exercise 2 passed! Tokens: {tokens}")

In [ ]:
# Exercise 3: Encode text with special tokens and return PyTorch tensors

tok_ex3 = AutoTokenizer.from_pretrained("bert-base-uncased")
encoded = tok_ex3("Test input", return_tensors=___)

assert isinstance(encoded['input_ids'], torch.Tensor)
# Should have [CLS] + test + input + [SEP] = 4 tokens
assert encoded['input_ids'].shape[1] == 4
print(f"Exercise 3 passed! Shape: {encoded['input_ids'].shape}")

In [ ]:
# Exercise 4: Get the [CLS] token embedding from BERT
# The [CLS] token is always at position 0

tok_ex4 = AutoTokenizer.from_pretrained("bert-base-uncased")
model_ex4 = AutoModel.from_pretrained("bert-base-uncased")

inputs = tok_ex4("Hello world", return_tensors="pt")
with torch.no_grad():
    outputs = model_ex4(**inputs)

# Extract the [CLS] embedding (batch index 0, token position 0)
cls_emb = outputs.last_hidden_state[___, ___]

assert cls_emb.shape == (768,)  # BERT-base hidden size
print(f"Exercise 4 passed! CLS embedding shape: {cls_emb.shape}")

In [ ]:
# Exercise 5: Convert model logits to probabilities

logits = torch.tensor([[2.5, -1.0]])  # fake logits for 2 classes

probs = torch.nn.functional.___(___, dim=-1)  # apply softmax to logits

assert abs(probs.sum().item() - 1.0) < 0.001
assert probs[0, 0] > probs[0, 1]
print(f"Exercise 5 passed! Probs: {probs}")

In [ ]:
# Exercise 6: Pad a batch of texts to the same length

tok_ex6 = AutoTokenizer.from_pretrained("bert-base-uncased")
texts_ex6 = ["Short.", "This is a longer sentence."]

batch = tok_ex6(texts_ex6, padding=___, truncation=True, return_tensors="pt")

assert batch['input_ids'].shape[0] == 2
assert batch['input_ids'][0].shape == batch['input_ids'][1].shape
print(f"Exercise 6 passed! Batch shape: {batch['input_ids'].shape}")

In [ ]:
# Exercise 7: Decode token IDs back to text

tok_ex7 = AutoTokenizer.from_pretrained("bert-base-uncased")
ids = tok_ex7.encode("Hello NLP world")

# Convert IDs back to text
decoded_text = tok_ex7.___(ids, skip_special_tokens=True)

assert "hello" in decoded_text.lower()
assert "nlp" in decoded_text.lower()
print(f"Exercise 7 passed! Decoded: '{decoded_text}'")

---
## Solutions

<details>
<summary>Exercise 1</summary>

```python
my_pipeline = pipeline("sentiment-analysis")
```
</details>

<details>
<summary>Exercise 2</summary>

```python
tokens = tok_ex2.tokenize("Hello world")
```
</details>

<details>
<summary>Exercise 3</summary>

```python
encoded = tok_ex3("Test input", return_tensors="pt")
```
</details>

<details>
<summary>Exercise 4</summary>

```python
cls_emb = outputs.last_hidden_state[0, 0]
```
</details>

<details>
<summary>Exercise 5</summary>

```python
probs = torch.nn.functional.softmax(logits, dim=-1)
```
</details>

<details>
<summary>Exercise 6</summary>

```python
batch = tok_ex6(texts_ex6, padding=True, truncation=True, return_tensors="pt")
```
</details>

<details>
<summary>Exercise 7</summary>

```python
decoded_text = tok_ex7.decode(ids, skip_special_tokens=True)
```
</details>

---
## Cumulative Review Exercises

Mixed exercises from Days 3-12. Fill in the blanks.

In [ ]:
# Review 1 (Day 3 - Data Cleaning): Remove duplicates from a list
data = [1, 2, 2, 3, 3, 3, 4]
unique = list(___(data))  # which built-in removes duplicates?

assert len(unique) == 4
print(f"Review 1 passed! Unique values: {sorted(unique)}")

In [ ]:
# Review 2 (Day 4 - Python Core): Dictionary comprehension
squares = {x: ___ for x in range(1, 6)}

assert squares == {1: 1, 2: 4, 3: 9, 4: 16, 5: 25}
print(f"Review 2 passed! {squares}")

In [ ]:
# Review 3 (Day 5 - PyTorch): Create a tensor and compute gradient
x = torch.tensor([3.0], requires_grad=___)
y = x ** 2  # y = x^2, dy/dx = 2x = 6
y.backward()

assert abs(x.grad.item() - 6.0) < 0.01
print(f"Review 3 passed! Gradient: {x.grad.item()}")

In [ ]:
# Review 4 (Day 6 - Pipelines): What does ColumnTransformer do?
answer_r4 = "___"  # "applies different transformations to different columns" or "transforms all columns the same way"?

assert answer_r4 == "applies different transformations to different columns"
print("Review 4 passed!")

In [ ]:
# Review 5 (Day 7 - Trees): What controls overfitting in decision trees?
answer_r5 = "___"  # name the most important hyperparameter

assert answer_r5 == "max_depth"
print("Review 5 passed!")

In [ ]:
# Review 6 (Day 8 - XGBoost): Random Forest uses ___, XGBoost uses ___
rf_approach = "___"      # "bagging" or "boosting"
xgb_approach = "___"     # "bagging" or "boosting"

assert rf_approach == "bagging"
assert xgb_approach == "boosting"
print("Review 6 passed!")

In [ ]:
# Review 7 (Day 9 - Metrics): Compute F1 score
precision_r7 = 0.8
recall_r7 = 0.6

f1 = 2 * (precision_r7 * recall_r7) / (precision_r7 ___ recall_r7)  # what operator?

assert abs(f1 - 0.6857) < 0.01
print(f"Review 7 passed! F1 = {f1:.4f}")

In [ ]:
# Review 8 (Day 10 - SHAP): SHAP values are based on which game theory concept?
answer_r8 = "___"  # "Shapley values" or "Nash equilibrium"

assert answer_r8 == "Shapley values"
print("Review 8 passed!")

In [ ]:
# Review 9 (Day 11 - TF-IDF): What does TF stand for?
answer_r9 = "___"  # spell it out

assert answer_r9.lower() == "term frequency"
print("Review 9 passed!")

In [ ]:
# Review 10 (Day 12 - Attention): Complete the attention formula
# Attention(Q, K, V) = softmax(Q @ K^T / sqrt(___)) @ V

d_k_review = 64
scale_factor = np.sqrt(___)

assert abs(scale_factor - 8.0) < 0.01  # sqrt(64) = 8
print(f"Review 10 passed! Scale factor: {scale_factor}")

### Cumulative Review Solutions

<details>
<summary>Review 1</summary>

```python
unique = list(set(data))
```
</details>

<details>
<summary>Review 2</summary>

```python
squares = {x: x**2 for x in range(1, 6)}
```
</details>

<details>
<summary>Review 3</summary>

```python
x = torch.tensor([3.0], requires_grad=True)
```
</details>

<details>
<summary>Review 4</summary>

```python
answer_r4 = "applies different transformations to different columns"
```
</details>

<details>
<summary>Review 5</summary>

```python
answer_r5 = "max_depth"
```
</details>

<details>
<summary>Review 6</summary>

```python
rf_approach = "bagging"
xgb_approach = "boosting"
```
</details>

<details>
<summary>Review 7</summary>

```python
f1 = 2 * (precision_r7 * recall_r7) / (precision_r7 + recall_r7)
```
</details>

<details>
<summary>Review 8</summary>

```python
answer_r8 = "Shapley values"
```
</details>

<details>
<summary>Review 9</summary>

```python
answer_r9 = "term frequency"
```
</details>

<details>
<summary>Review 10</summary>

```python
scale_factor = np.sqrt(d_k_review)
```
</details>

In [ ]:
# Cheat Sheet: Hugging Face Basics
cheat_sheet = """
=====================================================
         HUGGING FACE BASICS CHEAT SHEET
=====================================================

PIPELINE (quick & easy)
  pipe = pipeline("sentiment-analysis")
  pipe = pipeline("ner", grouped_entities=True)
  pipe = pipeline("fill-mask")
  pipe = pipeline("question-answering")
  pipe = pipeline("zero-shot-classification")
  pipe = pipeline("task", model="specific-model")

TOKENIZER
  tok = AutoTokenizer.from_pretrained("bert-base-uncased")
  tok.tokenize(text)              # split into tokens
  tok.encode(text)                # tokens -> IDs + specials
  tok(text, return_tensors="pt")  # recommended way
  tok.decode(ids)                 # IDs -> text
  tok(texts, padding=True, truncation=True)  # batching

MODEL
  model = AutoModel.from_pretrained("...")
    -> outputs.last_hidden_state  (batch, seq, hidden)
  model = AutoModelForSequenceClassification...
    -> outputs.logits  (batch, num_classes)

INFERENCE
  inputs = tok(text, return_tensors="pt")
  with torch.no_grad():
      outputs = model(**inputs)
  probs = torch.nn.functional.softmax(outputs.logits, -1)

KEY RULES
  1. Tokenizer and model MUST match
  2. Always use return_tensors="pt" for model input
  3. Use truncation=True for long texts
  4. Use torch.no_grad() for inference

=====================================================
"""
print(cheat_sheet)

---
## Next up: Day 14 -- FineTuningBERT

Tomorrow you'll take a pretrained BERT model and fine-tune it on your own labeled data using the Hugging Face Trainer API. You'll use `TrainingArguments`, the `datasets` library, and evaluate with `classification_report`.